# Bước 04/06 — Review KEEP/DROP + compose → `v6p5_compose_<12hex>`

**Không auto-promotion.** Mỗi ứng viên TRAIN cần `KEEP`/`DROP` trong `artifacts/qa/v6p5_review_decisions.csv` (trống = pending; file này được đồng bộ lên Camber). Còn pending → dừng ở `needs_review`. Exact-SHA trùng là hard gate; pHash phải review.

Quy trình: chạy → mở CSV điền decision/reviewer/comment → chạy lại.

| | Camber Stash |
|---|---|
| **Input** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/v6p5_ext_<12hex>/ (EXTERNAL_STAGE)` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/compose_qa/review_decisions/v6p5_ext_<12hex>/v6p5_review_decisions.csv` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/compose_qa/v6p5_compose_<12hex>/v6p5_compose_<12hex>.index.json  (field synthetic_archive)` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/compose_qa/v6p5_compose_<12hex>/v6p5_compose_<12hex>.freeze.json` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/compose_qa/v6p5_compose_<12hex>/v6p5_compose_<12hex>_synthetic_approved.tar.gz` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/compose_qa/v6p5_compose_<12hex>/*.review.resolved.csv, *.direct_approved_manifest.csv` |

Chạy lại notebook bất cứ lúc nào: bước đã xong + đúng config sẽ được **reuse**, bước dở dang sẽ **resume**.
Bước tiếp theo: **05_train_yolo11n_640_ablation**

In [ ]:
# Bootstrap: make the project package importable from notebooks/.
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("YOLOV11SDI_PROJECT_ROOT", str(ROOT))

import pandas as pd
from IPython.display import display, Image as IPImage
from yolov11sdi import pipeline
from yolov11sdi.config import load_config
from yolov11sdi.paths import ProjectPaths
from yolov11sdi.state import StateStore

FORCE_REBUILD = False   # True = rebuild even if a matching complete result exists
DRY_RUN = False
paths = ProjectPaths.detect().ensure()
store = StateStore(paths.state)
print("env:", paths.env.value, "| home:", paths.home, "| data:", paths.data_root)

In [ ]:
# ===== DATASET TRÊN CAMBER — điền tag nếu muốn chọn tay, None = tự động =====
PARENT_DATASET = "v6p2e_23562a45b343"   # V6.2E: cố định, KHÔNG đổi
EXTERNAL_STAGE = None   # vd "v6p5_ext_0123456789ab"     (tạo ở bước 03)
COMPOSE        = None   # vd "v6p5_compose_0123456789ab" (tạo ở bước 04)
REQUIRE_CAMBER = True   # lưu chính trên Camber -> dừng nếu chưa đăng nhập

from yolov11sdi.camber import CamberClient
from yolov11sdi.environment import load_camber_api_key
from yolov11sdi.reporting import status
cfg = pipeline.load(stage_tag=EXTERNAL_STAGE, compose_tag=COMPOSE, parent_tag=PARENT_DATASET)
load_camber_api_key(paths.env)
cam = CamberClient(paths, cfg.get("camber.stash_prefix"), cfg.runtime("camber_bin"))
if REQUIRE_CAMBER and not cam.available():
    raise RuntimeError("Camber chưa sẵn sàng: chạy `camber login` hoặc export CAMBER_API_KEY "
                       "(CAMBER_BIN=... nếu CLI không nằm trong PATH). Chỉ chạy local: REQUIRE_CAMBER=False")
display(status.dataset_card(cfg, paths, cam))

In [ ]:
outcome = pipeline.run_stage("compose", cfg, force=FORCE_REBUILD, dry_run=DRY_RUN)
print(outcome)
display(pd.Series(outcome.summary, dtype=object).to_frame("value"))

In [ ]:
dec_path = paths.resolve(cfg.get("compose.decisions_file"))
print("file review:", dec_path)
if dec_path.exists():
    dec = pd.read_csv(dec_path, keep_default_na=False)
    display(dec.assign(decision=dec["decision"].replace("", "PENDING"))
            .groupby(["candidate_type", "source", "target_split", "decision"]).size().rename("n").reset_index())
for name in ["pending_review.csv", "phash_candidates_to_review.csv"]:
    p = paths.compose / name
    if p.exists():
        print(name); display(pd.read_csv(p, keep_default_na=False).head(50))
for p in sorted((paths.working / "compose_qa_extract").glob("*/contacts/*.jpg"))[:cfg.get("compose.max_contact_sheets")]:
    print(p.name); display(IPImage(filename=str(p), width=640))

In [ ]:
# Kiểm tra output đã thật sự nằm trên Camber chưa
display(pipeline.verify_on_camber("compose", cfg))

Nếu trạng thái là `needs_review`: **dừng ở đây**, điền CSV rồi chạy lại notebook này.